# 🏃‍♂️ La Maratona Impossibile: Chi sopravvive ai 1000 Episodi?

**Data Storytelling a cura di [Il Tuo Nome]**

## Introduzione

Nel mondo degli anime, il dibattito sulla lunghezza è eterno. C'è chi sostiene che le serie lunghe (i cosiddetti "Long Runners" come *One Piece* o *Naruto*) soffrano inevitabilmente di un calo di qualità e di interesse pubblico, e chi invece ama immergersi in mondi infiniti.

In questa analisi, vogliamo sfatare o confermare il mito della **"Fatica dello Spettatore"**.

**Le Domande Chiave:**
1.  È vero che più una serie è lunga, più gente la abbandona (**Drop Rate**)?
2.  Come fanno titani come **One Piece** a violare le leggi della statistica?
3.  Esistono serie lunghe che sono state abbandonate in massa? E serie corte che hanno fallito subito?
4.  Il fenomeno dell'**On-Hold**: pausa tattica o abbandono silenzioso?

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

# Setup Stile Grafico
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

# Percorsi
DATA_DIR = Path("../../data/processed") # Adjust relative path as needed

def load_data(name):
    """Carica i dati parquet dalla cartella processed"""
    path = DATA_DIR / f"{name}.parquet"
    if not path.exists():
        # Fallback se eseguiamo da una posizione diversa
        path = Path("../../../data/processed") / f"{name}.parquet"
        if not path.exists():
            raise FileNotFoundError(f"Impossibile trovare {name}.parquet")
    return pd.read_parquet(path)

print("Librerie caricate e ambiente configurato.")

## 1. Preparazione dei Dati

Carichiamo i dataset `details` (info generali) e `stats` (statiche di visualizzazione) e creiamo la nostra metrica fondamentale: il **Drop Ratio**.

In [ ]:
# Caricamento Dati
df_details = load_data("details")
df_stats = load_data("stats")

# Merge
df = df_details.merge(df_stats, on="anime_id", how="inner")

# --- FEATURE ENGINEERING ---

# Calcolo Drop Ratio
# Definiamo il drop ratio come la percentuale di utenti che ha abbandonato rispetto a quelli che hanno interagito attivamente (Completed + Watching + Dropped + OnHold)
# Escludiamo 'Plan to Watch' perché spesso è una lista dei desideri passiva.
df['total_active'] = df['completed'] + df['watching'] + df['dropped'] + df['on_hold']
df['drop_ratio'] = (df['dropped'] / df['total_active']) * 100
df['on_hold_ratio'] = (df['on_hold'] / df['total_active']) * 100
df['completed_ratio'] = (df['completed'] / df['total_active']) * 100

# Categorie di Lunghezza
def categorize_length(episodes):
    if pd.isna(episodes):
        return "Unknown"
    if episodes <= 13:
        return "Corta (<=13)"
    elif episodes <= 26:
        return "Media (14-26)"
    elif episodes <= 100:
        return "Lunga (27-100)"
    else:
        return "Infinita (>100)"

df['length_cat'] = df['episodes'].apply(categorize_length)

# Ordine categorie per i grafici
cat_order = ["Corta (<=13)", "Media (14-26)", "Lunga (27-100)", "Infinita (>100)"]

# Filtri Pulizia
# Teniamo solo anime con un minimo di engagement per evitare rumore statistico
df_clean = df[(df['total_active'] > 500) & (df['episodes'] > 0)].copy()

print(f"Dataset pronto per l'analisi. Anime analizzati: {len(df_clean)}")
df_clean[['title', 'episodes', 'drop_ratio', 'length_cat']].head()

## Capitolo 1: La Curva della Fatica

Esiste una correlazione diretta tra il numero di episodi e la voglia di abbandonare? L'ipotesi è che richiedere 100 ore di tempo sia una barriera d'ingresso enorme.

In [ ]:
# Visualizzazione 1: Scatterplot Episodi vs Drop Ratio
plt.figure(figsize=(14, 8))

# Plot principale più pulito
sns.scatterplot(
    data=df_clean, 
    x='episodes', 
    y='drop_ratio', 
    alpha=0.6, 
    hue='length_cat', 
    palette='viridis', 
    size='members', 
    sizes=(20, 300),
    legend='brief'
)

plt.xscale('log')
plt.title('Il Paradosso: Serie più lunghe, meno abbandoni?', fontsize=18, pad=20)
plt.xlabel('Numero di Episodi (Scala Log)', fontsize=12)
plt.ylabel('Drop Ratio (%)', fontsize=12)

# Linea Media
mean_drop = df_clean['drop_ratio'].mean()
plt.axhline(y=mean_drop, color='gray', linestyle='--', alpha=0.7)
plt.text(1.5, mean_drop + 1, f'Media Drop: {mean_drop:.1f}%', color='gray')

# Annotazioni dirette per anime famosi (Evita di cercare nella legenda)
notable_anime = ['One Piece', 'Naruto', 'Bleach', 'Gintama', 'Death Note', 'Sword Art Online']
for title in notable_anime:
    row = df_clean[df_clean['title'].str.contains(title, regex=False)].head(1)
    if not row.empty:
        plt.text(
            row['episodes'].values[0], 
            row['drop_ratio'].values[0], 
            title, 
            fontsize=10, 
            fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1)
        )

sns.despine()
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
plt.tight_layout()
plt.show()

Notiamo subito una tendenza interessante: il tasso di abbandono non cresce linearmente. Ci sono molti anime corti con drop rate altissimi (rifiuto della qualità) e anime lunghissimi con drop rate bassi (selezione naturale dei fan).

Vediamo la distribuzione statistica tramite Boxplot.

In [ ]:
# Visualizzazione 2: Boxplot Drop Ratio per Categoria
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_clean, x='length_cat', y='drop_ratio', order=cat_order, palette='Set3')
plt.title('Distribuzione del Drop Ratio per Lunghezza Serie', fontsize=16)
plt.xlabel('Categoria Lunghezza', fontsize=12)
plt.ylabel('Drop Ratio (%)', fontsize=12)
plt.show()

Sorprendentemente, la categoria "Infinita (>100)" ha spesso una mediana di abbandono più **bassa** o comparabile alle serie medie. Chi inizia una serie di 500 episodi sa a cosa va incontro.

Influisce la popolarità? Le serie famose si droppano meno?

In [ ]:
# Visualizzazione 3: Hexbin Plot Popolarità vs Drop
plt.figure(figsize=(12, 6))
plt.hexbin(x=np.log1p(df_clean['members']), y=df_clean['drop_ratio'], gridsize=30, cmap='Blues', mincnt=1)
plt.colorbar(label='Numero di Anime')
plt.title('Popolarità (Members) vs Drop Ratio', fontsize=16)
plt.xlabel('Log(Members)', fontsize=12)
plt.ylabel('Drop Ratio (%)', fontsize=12)
plt.show()

## Capitolo 2: L'Anomalia "One Piece" (e i Giganti)

Analizziamo specificamente i "Mostri Sacri". Come si comportano i cosiddetti "Big Three" (One Piece, Naruto, Bleach) e altri giganti rispetto alla media?

In [ ]:
# Visualizzazione 4: Bar Chart Drop Ratio Top Long Runners
# Definiamo i Long Runners: Serie Infinite (>100 ep) con una fanbase rilevante
long_runners = df_clean[(df_clean['length_cat'] == 'Infinita (>100)') & (df_clean['members'] > 50000)].sort_values('drop_ratio')
long_runners = long_runners.head(20) # Prendiamo i primi 20 per leggibilità

plt.figure(figsize=(12, 8))

# Evidenziamo con colori: Rosso scuro per drop alto, Grigio per basso
colors = ['#e74c3c' if x > df_clean['drop_ratio'].mean() else '#95a5a6' for x in long_runners['drop_ratio']]
# Special color for One Piece
colors = ['#3498db' if 'One Piece' in t else c for t, c in zip(long_runners['title'], colors)]

barplot = sns.barplot(data=long_runners, x='drop_ratio', y='title', palette=colors)

plt.title('Drop Ratio: I Giganti a confronto', fontsize=18, pad=20)
plt.xlabel('Percentuale di Abbandono (%)', fontsize=12)
plt.ylabel('')

# Linea verticale media
plt.axvline(x=df_clean['drop_ratio'].mean(), color='black', linestyle=':', alpha=0.5)
plt.text(df_clean['drop_ratio'].mean()+0.5, 0, 'Media Globale', rotation=0, color='black', fontsize=10)

sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

**Gintama** e **Hunter x Hunter** mostrano tassi di abbandono incredibilmente bassi, segno di una qualità costante. **Bleach** e **Fairy Tail**, al contrario, soffrono di drop rate più alti (spesso a causa dei filler o archi narrativi ripetitivi).

C'è una relazione tra la qualità percepita (Score) e l'abbandono in queste serie lunghe?

In [ ]:
# Visualizzazione 5: Score vs Drop Ratio (Solo Long Runners)
long_runners_full = df_clean[df_clean['episodes'] > 50] # Prendiamo un campione più ampio

plt.figure(figsize=(12, 6))
sns.regplot(data=long_runners_full, x='score', y='drop_ratio', scatter_kws={'alpha':0.5}, line_kws={'color':'red'})
plt.title('Correlazione Qualità (Score) vs Abbandono nelle Serie Lunghe (>50 ep)', fontsize=16)
plt.xlabel('Score Medio', fontsize=12)
plt.ylabel('Drop Ratio (%)', fontsize=12)

# Annotiamo alcuni outlier
for i, row in long_runners.head(5).iterrows():
    plt.text(row['score'], row['drop_ratio'], row['title'], fontsize=9)

plt.show()

In [ ]:
# Visualizzazione 6: Composizione dell'Audience (Stacked Bar)
# Confronto diretto Big 3 + Gintama
selected_titles = ['One Piece', 'Naruto', 'Bleach', 'Gintama', 'Fairy Tail']
target_anime = df_clean[df_clean['title'].isin(selected_titles)].set_index('title')

# Nota: watching_ratio non era calcolato prima, lo aggiungiamo al volo o ricalcoliamo
target_anime['watching_ratio'] = (target_anime['watching'] / target_anime['total_active']) * 100

# Creiamo dataframe per lo stack
# ORDINE COLORI: Watching (Blue), Completed (Green), On-Hold (Yellow), Dropped (Red)
stack_data = target_anime[['watching_ratio', 'completed_ratio', 'on_hold_ratio', 'drop_ratio']]

# Palette semantica
colors = ['#3498db', '#2ecc71', '#f1c40f', '#e74c3c'] # Blue, Green, Yellow, Red

ax = stack_data.plot(kind='bar', stacked=True, figsize=(12, 7), color=colors, width=0.7)

plt.title('Come l\'utenza interagisce con i Giganti', fontsize=18, pad=20)
plt.ylabel('Percentuale Utenza (%)', fontsize=12)
plt.xlabel('')
plt.xticks(rotation=0)
plt.ylim(0, 100)

# Rimuoviamo bordi inutili
sns.despine(left=True)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.legend(title='Stato', bbox_to_anchor=(1.0, 1), loc='upper left', frameon=False)
plt.tight_layout()
plt.show()

## Capitolo 3: Il Cimitero dei Corti

Se la lunghezza non è l'unica causa di abbandono, allora cosa succede nelle serie corte? Qui il drop è spesso sinonimo di **rifiuto della qualità**.

Quali sono gli anime brevi che la gente ha abbandonato di più?

In [ ]:
# Visualizzazione 7: Top 10 Flop (Corti ma abbandonati)
all_time_flops = df_clean[(df_clean['length_cat'] == 'Corta (<=13)') & (df_clean['members'] > 10000)] # Filtro members per evitare anime sconosciuti
top_flops = all_time_flops.sort_values('drop_ratio', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_flops, y='title', x='drop_ratio', palette='magma')
plt.title('Il Cimitero dei Corti: Serie brevi con il più alto tasso di abbandono', fontsize=16)
plt.xlabel('Drop Ratio (%)', fontsize=12)
plt.ylabel('')
plt.show()

In [ ]:
# Visualizzazione 8: Score vs Drop per i Corti
plt.figure(figsize=(12, 6))
sns.scatterplot(data=all_time_flops, x='score', y='drop_ratio', alpha=0.3, color='purple')
plt.title('Qualità vs Abbandono nelle Serie Corte', fontsize=16)
plt.xlabel('Score', fontsize=12)
plt.ylabel('Drop Ratio (%)', fontsize=12)
plt.show()

La correlazione qui è evidente e brutale: se uno show corto ha uno score basso, viene abbandonato in massa immediatamente. Non c'è l'effetto "Sunk Cost Fallacy" (Fallacia dei costi irrecuperabili) che tiene la gente incollata alle serie lunghe.

## Capitolo 4: On-Hold, il Limbo

Le serie lunghe vengono davvero abbandonate, o vengono solo messe in pausa? L'On-Hold è il rifugio dello spettatore stanco che si promette "un giorno la finirò".

In [ ]:
# Visualizzazione 9: On-Hold vs Lunghezza
plt.figure(figsize=(12, 6))
sns.lineplot(data=df_clean, x='length_cat', y='on_hold_ratio', marker='o', sort=False, palette='coolwarm')
plt.title('Il Limbo: Tasso di On-Hold per Categoria di Lunghezza', fontsize=16)
plt.xlabel('Lunghezza', fontsize=12)
plt.ylabel('On-Hold Ratio (%)', fontsize=12)
plt.show()

In [ ]:
# Visualizzazione 10: Heatmap Correlazioni
corr_cols = ['episodes', 'score', 'members', 'drop_ratio', 'on_hold_ratio', 'completed_ratio']
corr_matrix = df_clean[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu', center=0, fmt=".2f")
plt.title('Matrice di Correlazione: Chi influenza l\'abbandono?', fontsize=16)
plt.show()

Notiamo che `episodes` ha una correlazione positiva con `on_hold_ratio`. Più è lunga la serie, più la gente la mette in pausa. Il `drop_ratio` ha una correlazione negativa forte con lo `score` (più è bello, meno lo mollo), ma debole con `episodes` (la lunghezza non è la causa primaria di abbandono).

## Capitolo 5: Stagionali vs Continui

L'industria è cambiata. Dal modello "One Piece" (settimanale continuo) al modello "Demon Slayer" (stagioni da 12/24 episodi). Questo ha ridotto la fatica?

In [ ]:
# Identifichiamo manualmente alcuni franchise rappresentativi
continuous_ids = ['One Piece', 'Naruto', 'Bleach', 'Black Clover', 'Dragon Ball Z', 'Fairy Tail']
seasonal_ids = ['Kimetsu no Yaiba', 'Boku no Hero Academia', 'Shingeki no Kyojin', 'Jujutsu Kaisen', 'Haikyuu!!']

# Funzione per taggare
def tag_format(title):
    for c in continuous_ids:
        if c in title: return "Continuo (Old School)"
    for s in seasonal_ids:
        if s in title: return "Stagionale (Moderno)"
    return "Altro"

df_clean['format_tag'] = df_clean['title'].apply(tag_format)
format_df = df_clean[df_clean['format_tag'] != "Altro"]

# Visualizzazione 11: Confronto Formati
plt.figure(figsize=(10, 6))
sns.barplot(data=format_df, x='format_tag', y='drop_ratio', ci=None, palette='pastel', estimator=np.median)
plt.title('Confronto Mediana Drop Ratio: Formato Continuo vs Stagionale', fontsize=16)
plt.ylabel('Mediana Drop Ratio (%)', fontsize=12)
plt.xlabel('')
plt.show()

In [ ]:
# Visualizzazione 12: Differenza per Tipo Media (TV vs ONA vs Movie)
# I Film si droppano meno delle serie?
plt.figure(figsize=(12, 6))
relevant_types = ['TV', 'Movie', 'OVA', 'ONA']
type_df = df_clean[df_clean['type'].isin(relevant_types)]

sns.violinplot(data=type_df, x='type', y='drop_ratio', palette='muted')
plt.title('Distribuzione Drop Ratio per Tipo di Media', fontsize=16)
plt.xlabel('Tipo', fontsize=12)
plt.ylabel('Drop Ratio (%)', fontsize=12)
plt.show()

## Conclusione: Il Mito della Fatica

Dai dati emerge una verità controintuitiva:

1.  **La lunghezza non uccide l'interesse**: Le serie "Infinite" (>100 ep) hanno tassi di abbandono spesso inferiori alle serie medie. Chi decide di scalarle, è motivato.
2.  **La qualità vince sempre**: Il `drop_ratio` è inversamente correlato allo `score` in modo netto. Una serie corta e brutta viene abbandonata molto più ferocemente di una serie lunga e mediocre.
3.  **One Piece è un'anomalia statistica**: Nonostante la lunghezza disumana, mantiene tassi di interazione altissimi, sfidando la curva della fatica.
4.  **Il Limbo dell'On-Hold**: Le serie lunghe non vengono uccise, vengono "congelate". L'alto rateo di On-Hold nelle serie >100 episodi suggerisce che gli utenti non rinunciano, ma prendono fiato.

In definitiva, la "Maratona Impossibile" è possibile, a patto che la storia valga la pena di essere vissuta.